# Prompt construction — SQL-Type query-plan generation

This notebook shows the **actual, executed result** of `build_query_plan_prompt`
(`src/model/Corelatte/QueryGeneration/SQL_TYPE_ALONE/prompt.py`) — the function
that builds the LLM prompt used by `SQL_TYPE_Generation.py` to generate each
benchmark query's plan.

It's built from three real MxFLS tables (`ii_portad`, `ii_vlh`, `ii_in`) plus
their codebook descriptions, exactly as the pipeline itself assembles them.
Two variants are shown: the **unbiased baseline** (`bias=None`, what v1 queries
use) and the **`having`-biased** variant (what v2 queries rotate through, along
with `scalar_filter`, `multi_join`, `column_provenance`, and `join_fanout`).
`random_seed=0` is passed for reproducibility — a real run leaves it `None`.

## Setup

Load the real tables and build the same `dataframe_description` text the
pipeline feeds into the prompt (table schemas + codebook column meanings).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.model.Corelatte.QueryGeneration.SQL_TYPE_ALONE import build_query_plan_prompt

RAW_DATA_ROOT = ROOT / "raw_data"
SOURCE_DATASET = "hh09dta_b2"
TABLE_NAMES = ["ii_portad", "ii_vlh", "ii_in"]


def load_tables(table_names):
    return {n: pd.read_stata(RAW_DATA_ROOT / SOURCE_DATASET / f"{n}.dta") for n in table_names}


def build_description(table_names):
    json_dir = RAW_DATA_ROOT / SOURCE_DATASET / "codebook_json"
    parts = []
    for t in table_names:
        text = (json_dir / f"{t}_enriched.txt").read_text(encoding="utf-8")
        parts.append(f"TABLE: {t}\nTEXT FILE: {t}_enriched.txt\n{text}")
    return "\n\n".join(parts)


tables = load_tables(TABLE_NAMES)
description = build_description(TABLE_NAMES)

for name, df in tables.items():
    print(f"{name:12s} {df.shape[0]:>6,} rows x {df.shape[1]} cols  -> {list(df.columns)}")
print()
print(f"dataframe_description: {len(description):,} chars")

ii_portad     9,092 rows x 5 cols  -> ['edad', 'ent', 'folio', 'ls', 'rel']
ii_vlh        9,092 rows x 64 cols  -> ['folio', 'vlh01a', 'vlh01b', 'vlh01c', 'vlh01d', 'vlh01e', 'vlh01f', 'vlh01g', 'vlh01h', 'vlh01i', 'vlh01j_1', 'vlh01k', 'vlh01l', 'vlh01m', 'vlh01n', 'vlh01o', 'vlh01p', 'vlh01q', 'vlh01r', 'vlh01s', 'vlh01t', 'vlh01u', 'vlh01v', 'vlh01w', 'vlh01x', 'vlh02_1', 'vlh02_2', 'vlh03', 'vlh04', 'vlh05', 'vlh06', 'vlh07a', 'vlh07b', 'vlh07c', 'vlh08a', 'vlh08b', 'vlh08c', 'vlh09a', 'vlh09b', 'vlh09c', 'vlh10a', 'vlh10b', 'vlh10c', 'vlh11a', 'vlh11b', 'vlh11c', 'vlh12a', 'vlh12a_a', 'vlh12a_b', 'vlh12a_c', 'vlh12b', 'vlh12c', 'vlh13', 'vlh13a', 'vlh14', 'vlh14a', 'vlh15', 'vlh15a', 'vlh16', 'vlh16a', 'vlh17', 'vlh17a', 'vlh18', 'vlh18a']
ii_in         9,092 rows x 65 cols  -> ['folio', 'in01a2_1', 'in01a2_2', 'in01a3_1', 'in01a3_2', 'in01a5_1', 'in01a5_2', 'in01a6_1', 'in01a6_2', 'in01a7_1', 'in01a7_2', 'in01a9_1', 'in01a9_2', 'in01a10_1', 'in01a10_2', 'in01a11_1', 'in01a11_2', 

## Baseline prompt (`bias=None`)

This is what `SQL_TYPE_Generation.py --version 1` sends to the model — no
structural bias injected.

In [2]:
prompt_baseline = build_query_plan_prompt(
    dataframes=tables,
    dataframe_description=description,
    number_of_nested_queries=2,
    n_sample_rows=3,
    bias=None,
    random_seed=0,
)

print(f"length: {len(prompt_baseline):,} chars\n")
print(prompt_baseline)

length: 38,571 chars

You are a query-planning assistant.

Your task is to generate a SQL-like query plan in JSON format.

The query plan will later be validated by a strict Pydantic schema and compiled into pandas code.

You are given:
1. A set of pandas DataFrames with their names, columns, dtypes, and sample rows.
2. A textual description explaining the semantic meaning of the DataFrames and their columns.
3. A required minimum number of nested query operations.

DATAFRAMES

{
  "name": "ii_portad",
  "n_rows": 9092,
  "columns": [
    "edad",
    "ent",
    "folio",
    "ls",
    "rel"
  ],
  "dtypes": {
    "edad": "float32",
    "ent": "float32",
    "folio": "object",
    "ls": "object",
    "rel": "float32"
  },
  "sample_rows": [
    {
      "edad": 44.0,
      "ent": 20.0,
      "folio": "000010AP00",
      "ls": "02",
      "rel": 20.0
    },
    {
      "edad": 24.0,
      "ent": 20.0,
      "folio": "000010BP03",
      "ls": "01",
      "rel": 20.0
    },
    {
      "edad

## Biased prompt (`bias="having"`)

Same DataFrames, same description — only the injected requirement block
differs. This is one of the five variants `--version 2` rotates through.

In [3]:
prompt_having = build_query_plan_prompt(
    dataframes=tables,
    dataframe_description=description,
    number_of_nested_queries=2,
    n_sample_rows=3,
    bias="having",
    random_seed=0,
)

print(f"length: {len(prompt_having):,} chars\n")
print(prompt_having)

length: 39,949 chars

You are a query-planning assistant.

Your task is to generate a SQL-like query plan in JSON format.

The query plan will later be validated by a strict Pydantic schema and compiled into pandas code.

You are given:
1. A set of pandas DataFrames with their names, columns, dtypes, and sample rows.
2. A textual description explaining the semantic meaning of the DataFrames and their columns.
3. A required minimum number of nested query operations.

DATAFRAMES

{
  "name": "ii_portad",
  "n_rows": 9092,
  "columns": [
    "edad",
    "ent",
    "folio",
    "ls",
    "rel"
  ],
  "dtypes": {
    "edad": "float32",
    "ent": "float32",
    "folio": "object",
    "ls": "object",
    "rel": "float32"
  },
  "sample_rows": [
    {
      "edad": 44.0,
      "ent": 20.0,
      "folio": "000010AP00",
      "ls": "02",
      "rel": 20.0
    },
    {
      "edad": 24.0,
      "ent": 20.0,
      "folio": "000010BP03",
      "ls": "01",
      "rel": 20.0
    },
    {
      "edad

## Prompt length across all five bias types

Same three tables and `random_seed=0` for every call, so the only thing
varying is the injected bias block.

In [4]:
BIAS_ROTATION = [None, "having", "scalar_filter", "multi_join", "column_provenance", "join_fanout"]

for bias in BIAS_ROTATION:
    p = build_query_plan_prompt(
        dataframes=tables,
        dataframe_description=description,
        number_of_nested_queries=2,
        n_sample_rows=3,
        bias=bias,
        random_seed=0,
    )
    label = bias if bias else "none (baseline)"
    print(f"{label:22s} {len(p):>7,} chars")

none (baseline)         38,571 chars
having                  39,949 chars
scalar_filter           39,761 chars
multi_join              40,055 chars
column_provenance       39,798 chars
join_fanout             40,306 chars